# Model Training using DistilBERT

In this notebook, we fine-tune the pretrained DistilBERT model on the BANKING77 dataset.

We use Transfer Learning instead of training a transformer from scratch because:

- Faster
- Better Accuracy
- Requires Less Data
- Lower Computational Cost

## Importing Libraries

In [2]:
import pandas as pd
import numpy as np
import torch
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score

In [3]:
# Checking Torch Version
print(torch.__version__)

2.13.0+cpu


In [4]:
#Gpu Check
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using Device :", device)

Using Device : cpu


## Loading Processed Data

In [5]:
train_df = pd.read_csv("../Data/train_processed.csv")

test_df = pd.read_csv("../Data/test_processed.csv")

In [6]:
train_df

,text,category,query_length,label
0,I am still waiting on my card?,card_arrival,30,12
1,What can I do if my card still hasn't arrived ...,card_arrival,60,12
2,I have been waiting over a week. Is the card s...,card_arrival,58,12
3,Can I track my card while it is in the process...,card_arrival,59,12
4,"How do I know if I will get my card, or if it ...",card_arrival,54,12
...,...,...,...,...
9998,You provide support in what countries?,country_support,38,25
9999,What countries are you supporting?,country_support,34,25
10000,What countries are getting support?,country_support,35,25
10001,Are cards available in the EU?,country_support,30,25


In [7]:
test_df

,text,category,query_length,label
0,How do I locate my card?,card_arrival,24,12
1,"I still have not received my new card, I order...",card_arrival,65,12
2,I ordered a card but it has not arrived. Help ...,card_arrival,53,12
3,Is there a way to know when my card will arrive?,card_arrival,48,12
4,My card has not arrived yet.,card_arrival,28,12
...,...,...,...,...
3075,"If i'm not in the UK, can I still get a card?",country_support,45,25
3076,How many countries do you support?,country_support,34,25
3077,What countries do you do business in?,country_support,37,25
3078,What are the countries you operate in.,country_support,38,25


In [8]:
# Load Tokenizer
tokenizer = DistilBertTokenizerFast.from_pretrained(
    "distilbert-base-uncased"
)

In [9]:
# Convert Pandas -> HuggingFace Dataset
from datasets import Dataset
train_dataset = Dataset.from_pandas(train_df)
train_dataset

Dataset({
    features: ['text', 'category', 'query_length', 'label'],
    num_rows: 10003
})

In [10]:
test_dataset = Dataset.from_pandas(test_df)
test_dataset

Dataset({
    features: ['text', 'category', 'query_length', 'label'],
    num_rows: 3080
})

In [11]:
# TOKENIZATION

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation = True,
        padding = "max_length",
        max_length = 64
    )

In [12]:
# Apply Tokenizer
train_dataset = train_dataset.map(
    tokenize,
    batched = True
)

test_dataset = test_dataset.map(
    tokenize,
    batched = True
)

Map:   0%|          | 0/10003 [00:00<?, ? examples/s]

Map:   0%|          | 0/3080 [00:00<?, ? examples/s]

In [13]:
# Removing Extra Cols
train_dataset = train_dataset.remove_columns(
    ["text","category","query_length"]
)
test_dataset = test_dataset.remove_columns(
    ["text","category","query_length"]
)

In [14]:
#Renaming label col
train_dataset = train_dataset.rename_column(
    "label",
    "labels"
)

test_dataset = test_dataset.rename_column(
    "label",
    "labels"
)

In [15]:
# In Pytorch format
train_dataset.set_format("torch")
test_dataset.set_format("torch")

In [16]:
train_dataset

Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 10003
})

In [17]:
test_dataset

Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 3080
})

In [18]:
# Load Model : DistilBERT

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels = 77
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [19]:
model.to(device)

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout(

In [20]:
pip install transformers[torch]

Note: you may need to restart the kernel to use updated packages.


In [21]:
# Training Args

training_args = TrainingArguments(
    output_dir = "models/saved_model",
    eval_strategy = "epoch", 
    save_strategy = "epoch",
    learning_rate = 2e-5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 16,
    num_train_epochs = 3,
    weight_decay = 0.01,
    logging_steps = 100,
    load_best_model_at_end = True,
    metric_for_best_model = "accuracy"
)

In [22]:
# Evaluation Function

def compute_metrics(pred):
    prediction = np.argmax(
        pred.predictions,
        axis = 1
    )

    accuracy = accuracy_score(
        pred.label_ids,
        prediction
    )

    return {
        "accuracy" : accuracy
    }

In [23]:
# Trainer

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = test_dataset,
    compute_metrics = compute_metrics
)
trainer.train()

C:\Users\Admin\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,2.265643,1.992214,0.715584
2,1.144333,1.058517,0.835714
3,0.820798,0.849728,0.859416


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\Admin\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\Admin\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1878, training_loss=1.825251972332549, metrics={'train_runtime': 13853.3592, 'train_samples_per_second': 2.166, 'train_steps_per_second': 0.136, 'total_flos': 497566386108288.0, 'train_loss': 1.825251972332549, 'epoch': 3.0})

In [24]:
trainer.evaluate()

C:\Users\Admin\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy
0.820798,0.849728,3,0.859416


{'eval_loss': 0.8497278094291687, 'eval_accuracy': 0.8594155844155844}

In [25]:
trainer.save_model("models/saved_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [26]:
tokenizer.save_pretrained(
    "models/saved_model"
)

('models/saved_model\\tokenizer_config.json',
 'models/saved_model\\tokenizer.json')

In [27]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model = AutoModelForSequenceClassification.from_pretrained("models/saved_model")
tokenizer = AutoTokenizer.from_pretrained("models/saved_model")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [28]:
history = trainer.state.log_history
history

[{'loss': 4.254554443359375,
  'grad_norm': 3.755857229232788,
  'learning_rate': 1.894568690095847e-05,
  'epoch': 0.1597444089456869,
  'step': 100},
 {'loss': 3.837513427734375,
  'grad_norm': 5.2450270652771,
  'learning_rate': 1.788072417465389e-05,
  'epoch': 0.3194888178913738,
  'step': 200},
 {'loss': 3.35986328125,
  'grad_norm': 5.9568634033203125,
  'learning_rate': 1.6815761448349307e-05,
  'epoch': 0.4792332268370607,
  'step': 300},
 {'loss': 2.946711120605469,
  'grad_norm': 5.645184516906738,
  'learning_rate': 1.5750798722044728e-05,
  'epoch': 0.6389776357827476,
  'step': 400},
 {'loss': 2.553294830322266,
  'grad_norm': 6.78502082824707,
  'learning_rate': 1.468583599574015e-05,
  'epoch': 0.7987220447284346,
  'step': 500},
 {'loss': 2.2656430053710936,
  'grad_norm': 6.254422187805176,
  'learning_rate': 1.3620873269435571e-05,
  'epoch': 0.9584664536741214,
  'step': 600},
 {'eval_loss': 1.9922140836715698,
  'eval_accuracy': 0.7155844155844155,
  'eval_runtime'